# Hyperparameter Tuning

## Goals

The overarching goal is to learn to use callbacks for some typical tasks. These include:
- Reporting about training progress.
- Stoping once training no longer reduces loss.
- Tuning hyperparameters.
- Implementing adaptive learning rate decay.
- Finding an optimal batch-size for training.
- Putting some of this into ```my_keras_utils.py``` so that they can be easily called and reused.

## What's Here?

I continue working with MNIST data, which I began working with in [my first Keras models](first_model.ipynb). 

My **concrete objective** is to tune a model that does well on Kaggle: 97th percentile? That's tough, but I think I can make it work.

In [1]:
import numpy as np
from datetime import datetime, time, timedelta

import pandas as pd
import tensorflow as tf
import kerastuner as kt
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt


import my_keras_utils as my_utils

In [2]:
tf.__version__
tf.config.experimental.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
## Load our data.
## Since the load process is a little slow, the try-except allows us to re-run all 
## cells without having to wait. 
try:
    ## Raises NameError and loads data if X_train is not defined.
    X_train.shape
except NameError: 
    ((X_train, y_train), (X_dev, y_dev), (X_test, y_test)) = my_utils.load_kaggle_mnist(dev_size=3000, 
    test_size=3000,
    random_state=42)

## Reshape the flattened rows into (-1,28,28,1)
## For augmentation
X_train = X_train.reshape((-1,28,28,1))    
X_dev = X_dev.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

In [4]:

def overlay_histories(histories, metric):
    fig, ax = plt.subplots()
    n = 0
    for h in histories:
        x = range(0,len(h.history[metric]))
        y = np.array(h.history[metric])
        label = 'history_' + str(n)
        ax.plot(x,y, label=label)
        n += 1
    ax.legend()

## Augmentation Model

In [5]:
augment_model = keras.models.Sequential(name= 'augment_layer')
augment_model.add(layers.experimental.preprocessing
                    .RandomRotation(factor = 1./20.,
                                    fill_mode = 'constant'))
augment_model.add(layers.experimental.preprocessing
                    .RandomTranslation(height_factor=2./28.,
                                    width_factor= 2./28.,
                                    fill_mode = 'constant'))
augment_model.add(layers.Flatten())

## Tuning

Time to work with Keras Tuner.



In [6]:
def model_builder(hp):

    inputs = keras.Input(shape=(28,28,1))
    x = augment_model(inputs)
    x = layers.experimental.preprocessing.Rescaling(1./255)(x)
    x = layers.Dropout(rate = hp.Float('drop_rate_0',
                                min_value = .0,
                                max_value = .15,
                                sampling = 'linear',
                                default = .0))(x)
    for i in range(hp.Int('num_layers', min_value = 2, max_value = 4, default = 2)):
        if i == 0:
            min_value = 100
            max_value = 160
        elif i == 3:
            min_value = 30
            max_value = 100
        else:
            min_value = 80
            max_value = 120
        hp_units = hp.Int('units_l'+str(i+1), 
                            min_value = min_value, 
                            max_value = max_value, 
                            default = min_value)
        x = layers.Dense(hp_units, activation = 'relu')(x)
        hp_dropout = hp.Float('drop_rate_'+str(i+1),
                                min_value = .0,
                                max_value = .15,
                                sampling = 'linear',
                                default = .0)
        x = layers.Dropout(rate=hp_dropout)(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer = keras.optimizers
                                .Adam(hp.Float('learning_rate',
                                                min_value = .00003,
                                                max_value = .01,
                                                sampling = 'log',
                                                default = .001)),
                    loss = "sparse_categorical_crossentropy",
                    run_eagerly = False,
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )

    return model

In [7]:
## Tuner callbacks
clear_output = my_utils.ClearTrainingOutput()
timed_update = my_utils.TimedProgressUpdate(3)
train_loss_stopping = keras.callbacks.EarlyStopping(monitor='loss', 
                            patience = 10, 
                            restore_best_weights = False
                            )
val_loss_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', 
                            patience = 10, 
                            restore_best_weights = False
                            )

adaptive_lr = keras.callbacks.ReduceLROnPlateau(
                    monitor='loss',
                    patience= 10,
                    cooldown= 2,
                    min_lr=.00005, 
                    factor=0.46416,)

## .46416 ~= .1**(1/3)

def schedule(epoch, lr):
    if epoch == 20:
        return lr * .46416
    if epoch == 50:
        return lr * .46416
    if epoch == 100:
        return lr * .46416
    return lr

scheduled_lr = keras.callbacks.LearningRateScheduler(schedule)

tuner_callbacks = [adaptive_lr, train_loss_stopping, val_loss_stopping, clear_output]

In [6]:

hp = kt.HyperParameters()
hp.Float('learning_rate',
            min_value = .00001,
            max_value = .0015,
            sampling = 'log',
            default = .0001)
hp.Fixed('num_layers', 3)

tuner = kt.BayesianOptimization(model_builder,
                objective = 'val_loss',
                hyperparameters= hp,
                max_trials = 100,
                num_initial_points = 5,
                executions_per_trial = 2,
                tune_new_entries = True,
                directory = os.path.normpath('C:\mnist'),
                project_name = 'lr',
                overwrite = False)
                
##assert 'keepgoing' == False
tuner.search(X_train,
                y_train,
                validation_data = (X_dev, y_dev),
                epochs = 200,
                batch_size = 32,
                verbose = 0,
                callbacks = [clear_output, val_loss_stopping])

INFO:tensorflow:Oracle triggered exit


In [8]:
tuner.results_summary()

In [9]:
fit_hp = kt.HyperParameters()
#fit_hp.Fixed('learning_rate', .002)
#fit_hp.Fixed('num_layers', 2)
model_2l = model_builder(kt.HyperParameters())
model_2l.summary()
#model_2l.get_config()

#assert False
history_l2 = model_2l.fit(X_train,
                y_train,
                epochs = 150,
                batch_size = 64,
                callbacks = [scheduled_lr],
                validation_data = (X_dev, y_dev),
                verbose = 2)


Model: "functional_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 28, 28, 1)]       0         
_________________________________________________________________
augment_layer (Sequential)   (None, 784)               0         
_________________________________________________________________
rescaling_1 (Rescaling)      (None, 784)               0         
_________________________________________________________________
dropout_3 (Dropout)          (None, 784)               0         
_________________________________________________________________
dense_3 (Dense)              (None, 100)               78500     
_________________________________________________________________
dropout_4 (Dropout)          (None, 100)               0         
_________________________________________________________________
dense_4 (Dense)              (None, 80)               

InternalError:  Blas GEMM launch failed : a.shape=(64, 784), b.shape=(784, 100), m=64, n=100, k=784
	 [[node functional_3/dense_3/MatMul (defined at <ipython-input-9-368db267ce7a>:15) ]] [Op:__inference_train_function_2060]

Function call stack:
train_function


In [ ]:
fit_hp.Fixed('num_layers', 3)
model_3l = model_builder(fit_hp)
#model_2l.summary()
#model_2l.get_config()


history_l3 = model_3l.fit(X_train,
                y_train,
                epochs = 150,
                batch_size = 64,
                callbacks = [scheduled_lr, clear_output],
                validation_data = (X_dev, y_dev),
                verbose = 0)

In [12]:
hp = kt.HyperParameters()
hp.Float('learning_rate',
            min_value = .0001,
            max_value = .0008,
            sampling = 'log')
#hp.Float('drop_rate_0', min_value=.0, max_value=.27)            
#hp.Float('drop_rate_1', min_value=.0, max_value=.15)
#hp.Float('drop_rate_2', min_value=.0, max_value=.15)
#hp.Float('drop_rate_3', min_value=.0, max_value=.15)
#hp.Fixed('drop_rate_0', 0)
hp.Int('num_layers', min_value = 2, max_value = 4)
#hp.Fixed('units_l1', 140)
#hp.Fixed('units_l2', 120)

tuner5 = kt.BayesianOptimization(model_builder,
                objective = 'loss',
                hyperparameters= hp,
                max_trials = 12,
                num_initial_points=3,
                executions_per_trial = 1,
                tune_new_entries = False,
                directory = os.path.normpath('C:\mnist'),
                project_name = 'cap2',
                overwrite = False)
                

#assert 'keepgoing' == False

tuner5.search(X_train,
                y_train,
                validation_data = (X_dev, y_dev),
                epochs = 100,
                batch_size = 64,
                verbose = 1,
                callbacks = [clear_output])

In [ ]:
#tuner2.search_space_summary()
tuner5.results_summary()

In [ ]:
|-drop_rate_0: 0.12293824152640359
|-drop_rate_1: 0.010395574155992644
|-drop_rate_2: 0.12864297589906631
|-drop_rate_3: 0.03723940790532728
|-learning_rate: 0.0007829421833955832
|-num_layers: 3
|-units_l1: 120
|-units_l2: 120
|-units_l3: 99

In [27]:

inputs = keras.Input(shape=(28,28,1))
x = augment_model(inputs)
x = layers.experimental.preprocessing.Rescaling(1./255)(x)
x = layers.Dropout(rate=.123)(x)
x = layers.Dense(120, activation='relu')(x)
x = layers.Dropout(rate=.010)(x)
x = layers.Dense(120, activation='relu')(x)
x = layers.Dropout(rate=.129)(x)
x = layers.Dense(99, activation='relu')(x)
x = layers.Dropout(rate=.037)(x)
output = layers.Dense(10, activation='softmax')(x)

mnist_model = keras.Model(name='MNIST_classifier', 
            inputs=inputs, 
            outputs=output)
optimizer = keras.optimizers.Adam(.0007829)
mnist_model.compile(optimizer=optimizer,
                loss = 'sparse_categorical_crossentropy',
                metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")])

mnist_model.summary()

Model: "MNIST_classifier"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         [(None, 28, 28, 1)]       0         
_________________________________________________________________
augment_layer (Sequential)   (None, 784)               0         
_________________________________________________________________
rescaling_2 (Rescaling)      (None, 784)               0         
_________________________________________________________________
dropout_8 (Dropout)          (None, 784)               0         
_________________________________________________________________
dense_8 (Dense)              (None, 120)               94200     
_________________________________________________________________
dropout_9 (Dropout)          (None, 120)               0         
_________________________________________________________________
dense_9 (Dense)              (None, 120)          

In [28]:
X = np.concatenate([X_train, X_dev])
y = np.concatenate([y_train, y_dev])
tensorboard = keras.callbacks.TensorBoard(log_dir='../../tb_logs')
csv_logger = keras.callbacks.CSVLogger(filename='../../csv_logs/mnist.csv', separator=',')
mnist_model.fit(X,
                y,
                epochs = 200,
                batch_size = 64,
                callbacks = [adaptive_lr, clear_output, tensorboard],
                validation_data = (X_test, y_test))

In [30]:
schedule = (lambda x: .00007)
lr_scheduler = keras.callbacks.LearningRateScheduler(schedule, verbose=0)
mnist_model.fit(X,
                y,
                epochs = 50,
                batch_size = 64,
                callbacks = [adaptive_lr],
                validation_data = (X_test, y_test))

Epoch 1/50
625/625 [==============================] - 4s 6ms/step - loss: 0.0421 - acc: 0.9860 - val_loss: 0.0387 - val_acc: 0.9880
Epoch 2/50
625/625 [==============================] - 4s 6ms/step - loss: 0.0420 - acc: 0.9866 - val_loss: 0.0390 - val_acc: 0.9880
Epoch 3/50
625/625 [==============================] - 4s 6ms/step - loss: 0.0410 - acc: 0.9858 - val_loss: 0.0395 - val_acc: 0.9875
Epoch 4/50
625/625 [==============================] - 4s 6ms/step - loss: 0.0404 - acc: 0.9869 - val_loss: 0.0396 - val_acc: 0.9875
Epoch 5/50
625/625 [==============================] - 4s 6ms/step - loss: 0.0409 - acc: 0.9865 - val_loss: 0.0397 - val_acc: 0.9875
Epoch 6/50
625/625 [==============================] - 4s 6ms/step - loss: 0.0387 - acc: 0.9870 - val_loss: 0.0400 - val_acc: 0.9880
Epoch 7/50
625/625 [==============================] - 4s 6ms/step - loss: 0.0402 - acc: 0.9867 - val_loss: 0.0398 - val_acc: 0.9890
Epoch 8/50
625/625 [==============================] - 4s 6ms/step - loss: 0.

In [15]:
%load_ext tensorboard

In [16]:
%tensorboard --logdir '../../tb_logs'

ERROR: Timed out waiting for TensorBoard to start. It may still be running as pid 16716.